# Gemma-Cyber v0.1: Free Cloud QLoRA Training & GGUF Export

This notebook fine-tunes **Gemma-3-4B-it** on the curated `sft_v0.1.jsonl` cybersecurity dataset using **QLoRA** (4-bit quantization + LoRA adapters), merges the weights, and exports a quantized **GGUF** model for local inference with Ollama.

### Hardware Requirement
* Free Google Colab **T4 GPU** (15GB VRAM) or **L4/A100**.

## Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q torch transformers peft trl bitsandbytes accelerate datasets pyyaml

## Step 2: Clone Gemma4-CyberAi Repository & Validate Dataset

In [ ]:
import json
from pathlib import Path

# If running in Colab without git clone, upload sft_v0.1.jsonl directly
dataset_path = 'data/training/sft_v0.1.jsonl'
if not Path(dataset_path).exists():
    print('Cloning repository...')
    !git clone https://github.com/novrusshehaj/Gemma4-CyberAi.git
    %cd Gemma4-CyberAi

# Verify dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    items = [json.loads(line) for line in f if line.strip()]
print(f'Successfully loaded {len(items)} training examples.')

## Step 3: Load Base Model with 4-bit Quantization (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = 'google/gemma-3-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: Train LoRA Adapter with SFTTrainer

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# Format chat messages
formatted = [{'messages': it['messages']} for it in items]
train_dataset = Dataset.from_list(formatted)

training_args = TrainingArguments(
    output_dir='./results_gemma3_cyber_v0.1',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_strategy='epoch',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained('./final_adapter')
tokenizer.save_pretrained('./final_adapter')
print('Training complete! Adapter saved to ./final_adapter')

## Step 5: Merge LoRA Adapter & Export to GGUF

In [ ]:
# Merge LoRA adapter into base model in FP16
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='cpu',
    trust_remote_code=True
)
merged_model = PeftModel.from_pretrained(base_model, './final_adapter')
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained('./gemma3-cyber-v0.1-merged')
tokenizer.save_pretrained('./gemma3-cyber-v0.1-merged')
print('Merged model saved.')

# Clone llama.cpp and convert to GGUF
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./gemma3-cyber-v0.1-merged --outfile gemma3-cyber-v0.1.gguf --outtype f16

# Quantize to Q4_K_M for local Ollama serving
!cd llama.cpp && make llama-quantize
!./llama.cpp/llama-quantize gemma3-cyber-v0.1.gguf gemma3-cyber-v0.1-Q4_K_M.gguf Q4_K_M
print('Exported gemma3-cyber-v0.1-Q4_K_M.gguf ready for Ollama!')

## Step 6: Create Ollama Model Locally

Download `gemma3-cyber-v0.1-Q4_K_M.gguf` to your local machine and run:
```bash
ollama create gemma3-cyber:v0.1 -f Modelfile.template
ollama run gemma3-cyber:v0.1 "Explain the MITRE ATT&CK technique for Kerberoasting."
```